## Step 1: Open the Dataset

In [3]:
import pandas as pd
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


## Step 2: Add Ground Truth Columns

In [ ]:

# 2. Define a helper function to guess the label based on keywords
def auto_label(text):
    text = str(text).lower()
    
    # Logic: Define your rules here
    if "security" in text or "alert" in text or "invoice" in text or "overdue" in text or "urgent" in text:
        return "notify", "urgent"
    elif "meeting" in text or "schedule" in text or "report" in text or "project" in text:
        return "respond", "neutral"
    elif "promotion" in text or "sale" in text or "newsletter" in text or "congratulations" in text:
        return "ignore", "neutral"
    elif "thanks" in text or "please" in text:
        return "ignore", "polite"
    else:
        return "respond", "neutral" # Default fallback

# 3. Apply the function to create new columns
df[['ideal_intent', 'ideal_tone']] = df['body'].apply(lambda x: pd.Series(auto_label(x)))

# 4. Save the updated file
df.to_csv("../data/sample_emails_with_triage_200.csv", index=False)

print("Success! File updated with 200 labels.")
print(df[['body', 'ideal_intent', 'ideal_tone']].head())

Success! File updated with 200 labels.
                                                body ideal_intent ideal_tone
0  Reminder: The client meeting is scheduled at 1...      respond    neutral
1  Your invoice of INR 25515.09 is due on 2025-12...       notify     urgent
2  Reminder: The client meeting is scheduled at 1...      respond    neutral
3  Hello team, please find the attached weekly re...      respond    neutral
4  Hello team, please find the attached weekly re...      respond    neutral


In [24]:
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,notify,urgent
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral


## Step 3: Email Assistant Logic (Reused from Milestone 1)

In [25]:
def email_assistant(email_text):
    text = email_text.lower()
    
    urgent_keywords = [
        "urgent", "overdue", "security", "breach", "declined", 
        "failed", "immediate", "server", "crash", "alert"
    ]
    ignore_keywords = [
        "newsletter", "automatic reply", "subscription", "deal", 
        "offer", "tracking", "system generated", "no reply", "marketing"
    ]
    if any(word in text for word in urgent_keywords):
        return "notify", "urgent"

    elif any(word in text for word in ignore_keywords):
        return "ignore", "neutral"
    
    elif "thank you" in text or "congratulations" in text:
        return "respond", "polite"

    else :
        return "respond", "neutral"

## Step 4: Generate Predictions

In [26]:
predictions = []
for _, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    predictions.append({
        "id": row["id"],
        "predicted_intent": action,
        "predicted_tone": tone
    })
pred_df = pd.DataFrame(predictions)
pred_df.head()

,id,predicted_intent,predicted_tone
0,1,respond,neutral
1,2,respond,neutral
2,3,respond,neutral
3,4,respond,neutral
4,5,respond,neutral


## Step 5: Evaluate Accuracy

In [27]:
def evaluate(row):
    score = 0
    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1
    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1
    return score

In [28]:
eval_df = df.merge(pred_df, on="id")
eval_df["score"] = eval_df.apply(evaluate, axis=1)
accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
accuracy

58.75

## Step 6: Save Evaluation Output

In [29]:
eval_df.to_csv(
    "../data/milestone2_output_Nitish_R_Maladakar.csv",
    index=False
)